# Treino com Extra Trees — 7 áreas (Whirlpool Thermal Predictor)

Treina o **Extra Trees Regressor** nas 7 áreas térmicas, usando:
- versão de losses: **13cat** (13 categorias)
- tratamento de nulos: **imputado_treino** (imputa só sobre o treino, sem vazamento)
- divisão treino/teste: **a partição única** gravada por `split_treino_teste`
- validação: **leave-one-group-out** sobre as configurações de treino

**Pré-requisitos na pasta:**
- `particao_treino_teste.csv` (gerado pelo notebook de split)
- `dataset_<area>_13cat.csv` para as 7 áreas (com os nulos ainda presentes — o tratamento acontece aqui)

> Trocar de experimento depois é fácil: mude `VERSAO` para `"perfil"` ou `TRATAMENTO` para `"mascara"`/`"imputado_base"` na seção 1.


## 1. Configuração

Aqui ficam as escolhas do experimento e as funções que vêm do notebook de split (copiadas para este notebook rodar sozinho). No projeto real, essas funções podem ser importadas do notebook de split em vez de copiadas.


In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- escolhas do experimento (troque aqui para rodar outra combinação) ---
VERSAO = "perfil"              # "13cat" ou "perfil"
TRATAMENTO = "imputado_treino"  # "mascara", "imputado_treino" ou "imputado_base"

AREAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
         "main_vent_sys", "mainback", "oven_hmi"]

# --- prefixos das colunas de entrada (features de design) ---
PREFIXOS_CONFIG = ("blower_direction_", "housing_", "board_position_",
                   "run_case_", "losses_", "perfil_perdas_",
                   "deflector_hole_", "material_coil_support_")

# limiares do tratamento de nulos (idênticos ao notebook de nulos e de split)
PCT_NULOS_MAX = 20.0
FAIXA_MAX = 35.0
MULT_EXTREMO = 2.5


## 2. Funções vindas do notebook de split

Estas funções padronizam como todos os experimentos consomem os dados, para que os resultados sejam comparáveis. São as mesmas do `split_treino_teste`.

- `colunas_de_configuracao`: quais colunas são entrada (X).
- `perfila_coluna` / `decide_acao`: as regras de tratamento de nulos.
- `monta_xy`: monta X e y de um componente, aplicando o tratamento escolhido — e nunca imputando no conjunto de avaliação.


In [ ]:
def colunas_de_configuracao(df):
    return [c for c in df.columns if c.startswith(PREFIXOS_CONFIG)]

def perfila_coluna(serie):
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    dentro = s[(s >= q1 - 1.5 * iqr) & (s <= q3 + 1.5 * iqr)]
    faixa = dentro.max() - dentro.min() if len(dentro) else 0.0
    extremos = int(((s <= q1 - MULT_EXTREMO * iqr) |
                    (s >= q3 + MULT_EXTREMO * iqr)).sum())
    return {"pct_nulos": 100 * serie.isna().sum() / len(serie),
            "faixa": faixa, "extremos": extremos,
            "media": s.mean(), "mediana": s.median()}

def decide_acao(perfil):
    if perfil["pct_nulos"] > PCT_NULOS_MAX and perfil["faixa"] > FAIXA_MAX:
        return "excluir"
    if perfil["extremos"] > 0:
        return "mediana"
    return "media"

def monta_xy(df, alvo, indices, tratamento, indices_treino=None):
    """Monta X e y de um componente restrito aos ensaios de 'indices'.
       A avaliação (validação/teste) deve sempre usar 'mascara'."""
    entrada = colunas_de_configuracao(df)
    sub = df.iloc[indices]

    if tratamento == "mascara":
        medido = sub[alvo].notna()
        return sub.loc[medido, entrada], sub.loc[medido, alvo]

    if tratamento == "imputado_base":       # arquivo _snull já vem preenchido
        return sub[entrada], sub[alvo]

    # imputado_treino: estatística calculada SÓ sobre o treino
    if indices_treino is None:
        raise ValueError("imputação exige indices_treino, para não vazar")
    perfil = perfila_coluna(df[alvo].iloc[indices_treino])
    acao = decide_acao(perfil)
    if acao == "excluir":
        return None, None
    return sub[entrada], sub[alvo].fillna(perfil[acao])


## 3. Carregar a partição e os dados

A partição diz, para cada ensaio (`run_idx`), se ele é treino ou teste e a qual dobra pertence. Lemos uma vez e usamos em todas as áreas — é o que garante que os 7 modelos usam a mesma divisão.


In [ ]:
particao = pd.read_csv("particao_treino_teste.csv")

idx_treino = particao.loc[particao["conjunto"] == "treino", "run_idx"].tolist()
idx_teste  = particao.loc[particao["conjunto"] == "teste",  "run_idx"].tolist()

print(f"Ensaios de treino: {len(idx_treino)}")
print(f"Ensaios de teste:  {len(idx_teste)}")
print(f"Dobras de validação (leave-one-group-out): "
      f"{particao.loc[particao.conjunto=='treino','dobra'].nunique()}")


## 4. Validação leave-one-group-out para um componente

A validação usa só o conjunto de **treino**. Cada dobra corresponde a uma configuração: deixamos essa configuração de fora, treinamos nas demais e prevemos nela. Isso simula prever uma configuração inédita.

Regras de método aplicadas aqui:
- O **treino** de cada dobra recebe o tratamento escolhido (imputação sobre o treino).
- A **avaliação** de cada dobra usa sempre `mascara` — nunca pontuamos contra valor imputado.
- Um componente cuja regra mande **excluir** devolve `None` e é pulado.


In [ ]:
def valida_componente(df, alvo, tratamento):
    """Leave-one-group-out sobre o treino. Devolve MAE, RMSE, R² e nº de pontos."""
    treino = particao[particao["conjunto"] == "treino"]
    dobras = sorted(treino["dobra"].dropna().astype(int).unique())

    y_real, y_prev = [], []
    for d in dobras:
        # índices (posição na tabela) de treino e de avaliação desta dobra
        idx_val = treino.loc[treino["dobra"] == d, "run_idx"].tolist()
        idx_fit = treino.loc[treino["dobra"] != d, "run_idx"].tolist()

        # treino: aplica o tratamento escolhido, usando só idx_fit como referência
        Xf, yf = monta_xy(df, alvo, idx_fit, tratamento, indices_treino=idx_fit)
        if Xf is None:            # coluna marcada para exclusão
            return None
        # avaliação: SEMPRE máscara (só linhas medidas)
        Xv, yv = monta_xy(df, alvo, idx_val, "mascara")
        if len(yv) == 0:          # componente não medido nesta dobra
            continue

        modelo = ExtraTreesRegressor(n_estimators=300, random_state=42)
        modelo.fit(Xf, yf)
        y_real.extend(yv.tolist())
        y_prev.extend(modelo.predict(Xv).tolist())

    if len(y_real) < 2:
        return None
    y_real, y_prev = np.array(y_real), np.array(y_prev)
    return {"n": len(y_real),
            "MAE":  mean_absolute_error(y_real, y_prev),
            "RMSE": np.sqrt(mean_squared_error(y_real, y_prev)),
            "R2":   r2_score(y_real, y_prev)}

# teste rápido num componente
demo = pd.read_csv(f"dataset_ipc_comp_{VERSAO}.csv")
print("Bridge:", valida_componente(demo, "Bridge", TRATAMENTO))


## 5. Avaliar uma área inteira

Roda a validação em todos os componentes de uma área e devolve uma tabela. Componentes excluídos pela regra de nulos ou sem dados suficientes são pulados (e contados).


In [ ]:
def avalia_area(area, versao, tratamento):
    df = pd.read_csv(f"dataset_{area}_{versao}.csv")
    entrada = set(colunas_de_configuracao(df))
    componentes = [c for c in df.columns if c not in entrada and c != "run_idx"]

    linhas, pulados = [], 0
    for comp in componentes:
        r = valida_componente(df, comp, tratamento)
        if r is None:
            pulados += 1
            continue
        linhas.append({"componente": comp, **r})
    return pd.DataFrame(linhas), pulados

tabela_ipc, pulados_ipc = avalia_area("ipc_comp", VERSAO, TRATAMENTO)
print(f"IPC Comp: {len(tabela_ipc)} componentes avaliados, {pulados_ipc} pulados")
tabela_ipc.sort_values("MAE").round(2)


## 6. Rodar as 7 áreas

Percorre todas as áreas e junta um resumo: MAE, RMSE e R² médios de cada uma. É o panorama geral do Extra Trees sob esta combinação (13cat + imputado_treino).


In [ ]:
resumo = []
detalhes = {}
for area in AREAS:
    tab, pulados = avalia_area(area, VERSAO, TRATAMENTO)
    detalhes[area] = tab
    resumo.append({
        "area": area,
        "componentes": len(tab),
        "pulados": pulados,
        "MAE":  tab["MAE"].mean(),
        "RMSE": tab["RMSE"].mean(),
        "R2":   tab["R2"].mean(),
    })

resumo = pd.DataFrame(resumo)
resumo.round(2)


## 7. Leitura dos resultados

Compare o MAE de cada área com o **piso de erro** dela (medido antes: de ~1,4 °C no Oven HMI a ~5,2 °C no IPC Comp). MAE perto do piso = modelo no limite do que os dados permitem. R² negativo = o modelo prevê pior que a média simples daquele componente, sinal de que faltam dados ou sinal para ele.

> **Importante (método):** estes números vêm da **validação** (leave-one-group-out sobre o treino), e servem para comparar opções — algoritmos, versões de losses, tratamentos de nulos. O **conjunto de teste** (`idx_teste`) permanece intocado e só deve ser usado uma vez, no final, para reportar o desempenho da configuração escolhida. Não use o teste para escolher entre experimentos.


In [ ]:
print(f"=== Extra Trees | losses={VERSAO} | nulos={TRATAMENTO} ===\n")
print(resumo.round(2).to_string(index=False))
print(f"\nMAE médio geral: {resumo['MAE'].mean():.2f} °C")
